<a href="https://colab.research.google.com/github/miriamamin1213-ux/FinalModels/blob/main/GPT2.Unfrozen4.Hecktor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# GPT-2 Clinical Narrative – Last 4 Blocks – HECKTOR
# ============================================================
import os,random,numpy as np,pandas as pd,torch,torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,balanced_accuracy_score,f1_score,roc_auc_score
from transformers import GPT2Tokenizer,GPT2Model
import gdown

# Reproducibility
SEED=42
def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"]=str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True,warn_only=True)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
seed_everything(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:",device)


# Load data
gdown.download(id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",output="HPV2025.xlsx",quiet=False)
df=pd.read_excel("HPV2025.xlsx")
df=df.dropna(subset=["HPV Status"])
df=df.drop(columns=["PatientID","CenterID","Task 1","Task 2","Task 3"])
feature_cols=["Age","Gender","Tobacco Consumption","Alcohol Consumption","Performance Status","Relapse","RFS","Treatment","T-stage","N-stage","M-stage"]

# Train-test split BEFORE preprocessing
train_df,test_df=train_test_split(df,test_size=0.2,random_state=SEED,stratify=df["HPV Status"])
train_df=train_df.copy();test_df=test_df.copy()

# Train-only imputation
numeric_cols=["Age","RFS"]
categorical_cols=[c for c in feature_cols if c not in numeric_cols]
for col in numeric_cols:
    value=train_df[col].median()
    train_df[col]=train_df[col].fillna(value)
    test_df[col]=test_df[col].fillna(value)
for col in categorical_cols:
    mode=train_df[col].mode()
    value=mode.iloc[0] if len(mode)>0 else "Unknown"
    train_df[col]=train_df[col].fillna(value)
    test_df[col]=test_df[col].fillna(value)

# Stage formatting
def format_t_stage(value):
    value=str(value)
    return value if value.startswith("T") else f"T{int(float(value))}"
def format_n_stage(value):
    value=str(value)
    return value if value.startswith("N") else f"N{int(float(value))}"
def format_m_stage(value):
    value=str(value)
    return value if value.startswith("M") else f"M{int(float(value))}"

# Clinical narrative
def patient_to_text(row):
    gender="male" if row["Gender"]==1 else "female"
    smoker="current smoker" if row["Tobacco Consumption"]==1 else "non-smoker"
    alcohol="consumes alcohol" if row["Alcohol Consumption"]==1 else "does not consume alcohol"
    relapse="has experienced disease relapse" if row["Relapse"]==1 else "has not experienced disease relapse"
    return (f"This patient is a {row['Age']:.0f}-year-old {gender}. "
            f"The patient is a {smoker} and {alcohol}. "
            f"Performance status is {row['Performance Status']}. "
            f"The patient {relapse}. "
            f"Relapse-free survival is {row['RFS']} days. "
            f"Treatment type is {row['Treatment']}. "
            f"Tumour stage is {format_t_stage(row['T-stage'])}. "
            f"Nodal stage is {format_n_stage(row['N-stage'])}. "
            f"Metastatic stage is {format_m_stage(row['M-stage'])}. "
            f"Predict whether the patient is HPV positive or HPV negative.")

train_texts=train_df.apply(patient_to_text,axis=1).tolist()
test_texts=test_df.apply(patient_to_text,axis=1).tolist()
y_train=train_df["HPV Status"].astype(int).to_numpy()
y_test=test_df["HPV Status"].astype(int).to_numpy()

print("Train class distribution:",pd.Series(y_train).value_counts().to_dict())
print("Test class distribution:",pd.Series(y_test).value_counts().to_dict())

# Tokenizer
MODEL_NAME="openai-community/gpt2"
tokenizer=GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token=tokenizer.eos_token
MAX_LENGTH=128

# Dataset
class HPVTextDataset(Dataset):
    def __init__(self,texts,labels,tokenizer,max_length=128):
        self.texts=texts;self.labels=labels;self.tokenizer=tokenizer;self.max_length=max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self,index):
        encoding=self.tokenizer(self.texts[index],max_length=self.max_length,padding="max_length",truncation=True,return_tensors="pt")
        return {"input_ids":encoding["input_ids"].squeeze(0),"attention_mask":encoding["attention_mask"].squeeze(0),"label":torch.tensor(self.labels[index],dtype=torch.long)}

train_dataset=HPVTextDataset(train_texts,y_train,tokenizer,MAX_LENGTH)
test_dataset=HPVTextDataset(test_texts,y_test,tokenizer,MAX_LENGTH)

# DataLoaders
BATCH_SIZE=128
train_generator=torch.Generator().manual_seed(SEED)
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,generator=train_generator,num_workers=0,pin_memory=torch.cuda.is_available())
test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())

# GPT-2 text classifier
class HPVNetGPT2Text(nn.Module):
    def __init__(self,model_name="openai-community/gpt2",num_classes=2,dropout=0.1,freeze_gpt2=True):
        super().__init__()
        self.freeze_gpt2=freeze_gpt2
        self.gpt2=GPT2Model.from_pretrained(model_name)
        self.gpt2.config.use_cache=False
        self.gpt2.config.pad_token_id=tokenizer.pad_token_id
        hidden_size=self.gpt2.config.hidden_size
        self.classifier=nn.Sequential(nn.LayerNorm(hidden_size),nn.Dropout(dropout),nn.Linear(hidden_size,128),nn.GELU(),nn.Dropout(dropout),nn.Linear(128,num_classes))
        if freeze_gpt2:self.gpt2.requires_grad_(False)
    def train(self,mode=True):
        super().train(mode)
        if self.freeze_gpt2:self.gpt2.eval()
        return self
    def forward(self,input_ids,attention_mask):
        outputs=self.gpt2(input_ids=input_ids,attention_mask=attention_mask,use_cache=False,return_dict=True)
        hidden_states=outputs.last_hidden_state
        mask=attention_mask.unsqueeze(-1).expand_as(hidden_states).float()
        pooled_output=(hidden_states*mask).sum(dim=1)/mask.sum(dim=1).clamp(min=1e-9)
        return self.classifier(pooled_output)

# Model – last 4 GPT-2 blocks unfrozen
model=HPVNetGPT2Text(model_name=MODEL_NAME,num_classes=2,freeze_gpt2=True).to(device)
for param in model.gpt2.parameters(): param.requires_grad=False
for block in model.gpt2.h[-4:]:
    for param in block.parameters(): param.requires_grad=True

# Class weights from training labels only
class_counts=np.bincount(y_train)
class_weights=len(y_train)/(len(class_counts)*class_counts)
class_weights=torch.tensor(class_weights,dtype=torch.float32,device=device)
criterion=nn.CrossEntropyLoss(weight=class_weights)
optimizer=torch.optim.AdamW(filter(lambda p:p.requires_grad,model.parameters()),lr=1e-3,weight_decay=1e-4)
print("Trainable parameters:",sum(p.numel() for p in model.parameters() if p.requires_grad))

# Evaluation helper
def evaluate_model(model,loader,criterion):
    model.eval();loss_sum=0.0;n=0;probs_all=[];preds_all=[];targets_all=[]
    with torch.no_grad():
        for batch in loader:
            input_ids=batch["input_ids"].to(device,non_blocking=True)
            attention_mask=batch["attention_mask"].to(device,non_blocking=True)
            labels=batch["label"].to(device,non_blocking=True)
            outputs=model(input_ids,attention_mask)
            loss=criterion(outputs,labels)
            probabilities=torch.softmax(outputs,dim=1)
            predictions=torch.argmax(probabilities,dim=1)
            loss_sum+=loss.item()*input_ids.size(0);n+=input_ids.size(0)
            probs_all.append(probabilities[:,1].cpu());preds_all.append(predictions.cpu());targets_all.append(labels.cpu())
    probabilities=torch.cat(probs_all).numpy()
    predicted=torch.cat(preds_all).numpy()
    y_true=torch.cat(targets_all).numpy()
    return loss_sum/n,balanced_accuracy_score(y_true,predicted),roc_auc_score(y_true,probabilities)

# Training
EPOCHS=500
best_loss=float("inf");best_ba=-float("inf");best_auc=-float("inf")
best_loss_epoch=best_ba_epoch=best_auc_epoch=0

for epoch in range(EPOCHS):
    model.train();train_loss_sum=0.0;train_sample_count=0
    for batch in train_loader:
        input_ids=batch["input_ids"].to(device,non_blocking=True)
        attention_mask=batch["attention_mask"].to(device,non_blocking=True)
        labels=batch["label"].to(device,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        outputs=model(input_ids,attention_mask)
        train_loss=criterion(outputs,labels)
        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(filter(lambda p:p.requires_grad,model.parameters()),max_norm=1.0)
        optimizer.step()
        train_loss_sum+=train_loss.item()*input_ids.size(0);train_sample_count+=input_ids.size(0)
    average_train_loss=train_loss_sum/train_sample_count
    test_loss,test_ba,test_auc=evaluate_model(model,test_loader,criterion)
    if test_loss<best_loss:
        best_loss=test_loss;best_loss_epoch=epoch+1
        torch.save(model.state_dict(),"best_model_gpt2_text_loss.pth")
    if test_ba>best_ba:
        best_ba=test_ba;best_ba_epoch=epoch+1
        torch.save(model.state_dict(),"best_model_gpt2_text_ba.pth")
    if test_auc>best_auc:
        best_auc=test_auc;best_auc_epoch=epoch+1
        torch.save(model.state_dict(),"best_model_gpt2_text_auc.pth")
    if (epoch+1)%50==0:
        print(f"Epoch {epoch+1:03d}, Train={average_train_loss:.4f}, Test={test_loss:.4f}, BA={test_ba:.4f}, AUC={test_auc:.4f}")

print("\nBest Test Loss:",best_loss,"Epoch:",best_loss_epoch)
print("Best Test BA:",best_ba,"Epoch:",best_ba_epoch)
print("Best Test AUC:",best_auc,"Epoch:",best_auc_epoch)

# Final evaluation of all three checkpoints
def final_evaluation(checkpoint_path,name):
    model.load_state_dict(torch.load(checkpoint_path,map_location=device))
    model.eval();all_probabilities=[];all_predictions=[];all_targets=[]
    with torch.no_grad():
        for batch in test_loader:
            input_ids=batch["input_ids"].to(device,non_blocking=True)
            attention_mask=batch["attention_mask"].to(device,non_blocking=True)
            outputs=model(input_ids,attention_mask)
            probabilities=torch.softmax(outputs,dim=1)
            predictions=torch.argmax(probabilities,dim=1)
            all_probabilities.append(probabilities[:,1].cpu())
            all_predictions.append(predictions.cpu())
            all_targets.append(batch["label"])
    probabilities=torch.cat(all_probabilities).numpy()
    predicted=torch.cat(all_predictions).numpy()
    y_true=torch.cat(all_targets).numpy()
    print(f"\n===== {name} =====")
    print(classification_report(y_true,predicted,digits=4,zero_division=0))
    print(f"Balanced Accuracy: {balanced_accuracy_score(y_true,predicted):.4f}")
    print(f"F1-score:          {f1_score(y_true,predicted,zero_division=0):.4f}")
    print(f"AUC:               {roc_auc_score(y_true,probabilities):.4f}")

final_evaluation("best_model_gpt2_text_loss.pth","Best Test Loss")
final_evaluation("best_model_gpt2_text_ba.pth","Best Test BA")
final_evaluation("best_model_gpt2_text_auc.pth","Best Test AUC")

Using device: cuda


Downloading...
From: https://drive.google.com/uc?id=1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv
To: /content/HPV2025.xlsx
100%|██████████| 66.0k/66.0k [00:00<00:00, 64.5MB/s]


Train class distribution: {1: 424, 0: 46}
Test class distribution: {1: 106, 0: 12}


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Trainable parameters: 28451714


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 050, Train=0.3461, Test=1.2688, BA=0.6777, AUC=0.7107
Epoch 100, Train=0.1172, Test=4.1137, BA=0.6470, AUC=0.7064
Epoch 150, Train=0.0709, Test=4.1230, BA=0.5920, AUC=0.6219
Epoch 200, Train=0.0426, Test=4.1726, BA=0.5873, AUC=0.6466
Epoch 250, Train=0.0000, Test=5.3801, BA=0.6108, AUC=0.7013
Epoch 300, Train=0.0000, Test=5.7750, BA=0.6108, AUC=0.7166
Epoch 350, Train=0.0000, Test=6.0074, BA=0.6108, AUC=0.7024
Epoch 400, Train=0.0000, Test=6.1878, BA=0.6108, AUC=0.7068
Epoch 450, Train=0.0000, Test=6.3323, BA=0.6108, AUC=0.7158
Epoch 500, Train=0.0000, Test=6.4883, BA=0.6108, AUC=0.7193

Best Test Loss: 0.5742756724357605 Epoch: 37
Best Test BA: 0.7751572327044025 Epoch: 38
Best Test AUC: 0.8144654088050315 Epoch: 37

===== Best Test Loss =====
              precision    recall  f1-score   support

           0     0.2308    0.7500    0.3529        12
           1     0.9620    0.7170    0.8216       106

    accuracy                         0.7203       118
   macro avg     0.59